Please write a complete Python script that implements the classic CART (Classification and Regression Trees) algorithm entirely from scratch (without using scikit-learn or any other machine learning frameworks for the model itself).

For the dataset, please use the COMPASS dataset from Kaggle. Use the following kagglehub snippet to fetch the data:

In [1]:
import kagglehub
path = kagglehub.dataset_download("danofer/compass")
print("Path to dataset files:", path)

/opt/anaconda3/envs/dataScience/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/jeanniecheng/.cache/kagglehub/datasets/danofer/compass/versions/1


Requirements:
1.	Data Loading & Preprocessing: Locate the main .csv file in the downloaded path and load it using pandas. Clean the data and select a subset of relevant features to keep the custom tree implementation performant. Define the target variable (e.g., is_recid).

2.	Custom CART Implementation: Create a custom Python class for the Decision Tree. It must include the logic for calculating Gini impurity, finding the best splits for both numerical and categorical data, and recursively building the tree up to a specified max_depth.

3.	Training & Testing: Split the dataset into train/test sets, train your custom CART model, and make predictions on the test set.

4.	Evaluation: Calculate and print the final accuracy score.

5.	Please ensure the code is fully self-contained, runnable, and heavily commented to explain the core CART logic.

6. If I dont have the required packages, help me automatically install them in conda dataScience
   

In [2]:
# ============================================================
# Imports & data download
# ============================================================
import os
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Download COMPASS dataset
path = kagglehub.dataset_download("danofer/compass")
print("Path to dataset files:", path)
print("Files:", os.listdir(path))

Path to dataset files: /Users/jeanniecheng/.cache/kagglehub/datasets/danofer/compass/versions/1
Files: ['cox-violent-parsed_filt.csv', 'compas-scores-raw.csv', 'cox-violent-parsed.csv', 'propublicaCompassRecividism_data_fairml.csv']


In [3]:
# ============================================================
# Data loading & preprocessing
# ============================================================
# Locate the main CSV file. We use cox-violent-parsed_filt.csv since it
# contains the cleaned, filtered ProPublica COMPAS records with `is_recid`.
csv_path = os.path.join(path, "cox-violent-parsed_filt.csv")
df = pd.read_csv(csv_path)
print("Raw shape:", df.shape)

# Select a small set of features that are meaningful for recidivism prediction.
# Keep the feature set small so the pure-Python tree stays fast.
NUMERIC_FEATURES = [
    "age",
    "priors_count",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
]
CATEGORICAL_FEATURES = [
    "sex",
    "race",
    "c_charge_degree",
]
TARGET = "is_recid"

cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]
data = df[cols].copy()

# `is_recid == -1` means missing recidivism status -> drop those rows.
data = data[data[TARGET].isin([0, 1])]

# Drop any rows with missing feature values.
data = data.dropna().reset_index(drop=True)
print("Cleaned shape:", data.shape)
print("Class balance:\n", data[TARGET].value_counts(normalize=True))

X = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = data[TARGET].astype(int).values

Raw shape: (18316, 40)
Cleaned shape: (17448, 9)
Class balance:
 is_recid
0    0.52006
1    0.47994
Name: proportion, dtype: float64


In [4]:
# ============================================================
# Unified CART (classification + regression) — from scratch
# ============================================================
# A single tree class parameterised by `task`:
#   - "classification": Gini impurity, majority-vote leaves, supports
#                       binary AND multiclass targets.
#   - "regression":     variance impurity (MSE), mean-value leaves.
#
# Categorical splits use Breiman's classical shortcut:
#   * Binary classification: sort categories by P(y=1 | category)
#   * Regression:            sort categories by E[y | category]
#   * Multiclass:            no closed-form prefix shortcut, so we either
#                            brute-force subsets when K is small, or skip.

from itertools import combinations


class Node:
    """A single tree node — either internal (with a split) or a leaf."""
    __slots__ = ("feature", "is_numeric", "threshold", "left_categories",
                 "left", "right", "prediction")

    def __init__(self):
        self.feature = None
        self.is_numeric = None
        self.threshold = None
        self.left_categories = None
        self.left = None
        self.right = None
        self.prediction = None     # class label OR mean target value


class CART:
    """CART for classification (binary/multiclass) or regression."""

    def __init__(self, task="classification", max_depth=6,
                 min_samples_split=20, min_samples_leaf=10,
                 numeric_features=None, categorical_features=None,
                 max_cat_brute_force=10):
        assert task in ("classification", "regression")
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.numeric_features = list(numeric_features or [])
        self.categorical_features = list(categorical_features or [])
        # If a categorical feature has more distinct levels than this and the
        # problem is multiclass, we skip it rather than enumerate 2^K subsets.
        self.max_cat_brute_force = max_cat_brute_force
        self.root = None
        self._classes = None  # populated in fit() for classification

    # ---------- Impurity ----------
    def _impurity(self, y):
        """Gini for classification, variance for regression."""
        if len(y) == 0:
            return 0.0
        if self.task == "classification":
            _, counts = np.unique(y, return_counts=True)
            p = counts / counts.sum()
            return 1.0 - np.sum(p * p)
        # regression: population variance acts as MSE around the mean
        return float(np.var(y))

    def _weighted_impurity(self, y_left, y_right):
        n = len(y_left) + len(y_right)
        return (len(y_left) / n) * self._impurity(y_left) + \
               (len(y_right) / n) * self._impurity(y_right)

    # ---------- Leaf prediction ----------
    def _leaf_value(self, y):
        if self.task == "classification":
            vals, counts = np.unique(y, return_counts=True)
            return vals[np.argmax(counts)]
        return float(np.mean(y))

    # ---------- Numeric split ----------
    def _best_numeric_split(self, column, y):
        """Sweep candidate thresholds in O(N log N).

        Classification uses cumulative class counts; regression uses
        cumulative sums / sums-of-squares so variance at every cut is
        constant-time.
        """
        values = np.asarray(column.values)
        order = np.argsort(values, kind="mergesort")
        v_sorted = values[order]
        y_sorted = np.asarray(y)[order]
        n = len(y_sorted)
        if n < 2 * self.min_samples_leaf:
            return 0.0, None

        parent_imp = self._impurity(y_sorted)
        best_gain = 0.0
        best_threshold = None

        if self.task == "classification":
            classes = self._classes
            class_to_idx = {c: i for i, c in enumerate(classes)}
            y_idx = np.array([class_to_idx[v] for v in y_sorted])
            cum = np.zeros((n + 1, len(classes)), dtype=np.int64)
            for i in range(n):
                cum[i + 1] = cum[i]
                cum[i + 1, y_idx[i]] += 1
            total = cum[-1]

            for i in range(self.min_samples_leaf, n - self.min_samples_leaf + 1):
                if v_sorted[i] == v_sorted[i - 1]:
                    continue
                left_counts = cum[i]
                right_counts = total - left_counts
                n_left, n_right = i, n - i
                p_left = left_counts / n_left
                p_right = right_counts / n_right
                g_left = 1.0 - np.sum(p_left * p_left)
                g_right = 1.0 - np.sum(p_right * p_right)
                w_imp = (n_left / n) * g_left + (n_right / n) * g_right
                gain = parent_imp - w_imp
                if gain > best_gain:
                    best_gain = gain
                    best_threshold = (v_sorted[i - 1] + v_sorted[i]) / 2.0
        else:
            # Regression: variance via cumulative sum and sum-of-squares.
            cs = np.concatenate([[0.0], np.cumsum(y_sorted)])
            css = np.concatenate([[0.0], np.cumsum(y_sorted * y_sorted)])
            total_s, total_ss = cs[-1], css[-1]

            for i in range(self.min_samples_leaf, n - self.min_samples_leaf + 1):
                if v_sorted[i] == v_sorted[i - 1]:
                    continue
                n_left, n_right = i, n - i
                s_left, ss_left = cs[i], css[i]
                s_right, ss_right = total_s - s_left, total_ss - ss_left
                var_left = ss_left / n_left - (s_left / n_left) ** 2
                var_right = ss_right / n_right - (s_right / n_right) ** 2
                w_imp = (n_left / n) * var_left + (n_right / n) * var_right
                gain = parent_imp - w_imp
                if gain > best_gain:
                    best_gain = gain
                    best_threshold = (v_sorted[i - 1] + v_sorted[i]) / 2.0
        return best_gain, best_threshold

    # ---------- Categorical split ----------
    def _ordered_categories(self, values, y, categories):
        """Order categories so the optimal split is some prefix.

        - Binary classification: order by P(y=1 | category)
        - Regression:            order by E[y | category]
        Both are Breiman's provably-optimal orderings.
        """
        keys = []
        for c in categories:
            mask = values == c
            keys.append((c, y[mask].mean() if mask.sum() else 0.0))
        keys.sort(key=lambda t: t[1])
        return [c for c, _ in keys]

    def _best_categorical_split(self, column, y):
        values = np.asarray(column.values)
        categories = np.unique(values)
        if len(categories) <= 1:
            return 0.0, None

        is_binary_clf = (self.task == "classification"
                         and len(self._classes) == 2)
        is_regression = (self.task == "regression")

        # Candidate "left" subsets to try.
        if is_binary_clf or is_regression:
            sorted_cats = self._ordered_categories(values, y, categories)
            # Optimal split is a prefix of the sorted order — O(K log K).
            candidate_left_sets = [set(sorted_cats[:k])
                                   for k in range(1, len(sorted_cats))]
        else:
            # Multiclass: brute-force all non-trivial subsets when small,
            # otherwise skip to keep training tractable.
            K = len(categories)
            if K > self.max_cat_brute_force:
                return 0.0, None
            cat_list = list(categories)
            candidate_left_sets = []
            # Enumerate halves once (a subset and its complement give the
            # same partition, so we stop at K/2).
            for r in range(1, K // 2 + 1):
                for combo in combinations(cat_list, r):
                    candidate_left_sets.append(set(combo))

        parent_imp = self._impurity(y)
        best_gain = 0.0
        best_left = None
        n = len(y)

        for left_set in candidate_left_sets:
            left_mask = np.array([v in left_set for v in values])
            n_left = left_mask.sum()
            n_right = n - n_left
            if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                continue
            w_imp = self._weighted_impurity(y[left_mask], y[~left_mask])
            gain = parent_imp - w_imp
            if gain > best_gain:
                best_gain = gain
                best_left = left_set
        return best_gain, best_left

    def _best_split(self, X, y):
        best = {"gain": 0.0}
        for feat in self.numeric_features:
            gain, thr = self._best_numeric_split(X[feat], y)
            if gain > best["gain"] and thr is not None:
                best = {"gain": gain, "feature": feat, "is_numeric": True,
                        "threshold": thr}
        for feat in self.categorical_features:
            gain, left_set = self._best_categorical_split(X[feat], y)
            if gain > best["gain"] and left_set is not None:
                best = {"gain": gain, "feature": feat, "is_numeric": False,
                        "left_categories": left_set}
        return best if best["gain"] > 0 else None

    # ---------- Recursive build ----------
    def _build(self, X, y, depth):
        node = Node()
        # Stop if pure (classification) / zero-variance (regression),
        # at depth limit, or below the minimum split size.
        if (depth >= self.max_depth
                or len(y) < self.min_samples_split
                or self._impurity(y) == 0.0):
            node.prediction = self._leaf_value(y)
            return node

        split = self._best_split(X, y)
        if split is None:
            node.prediction = self._leaf_value(y)
            return node

        node.feature = split["feature"]
        node.is_numeric = split["is_numeric"]
        if node.is_numeric:
            node.threshold = split["threshold"]
            left_mask = X[node.feature].values <= node.threshold
        else:
            node.left_categories = split["left_categories"]
            left_mask = np.array([v in node.left_categories
                                  for v in X[node.feature].values])

        X_left = X[left_mask].reset_index(drop=True)
        X_right = X[~left_mask].reset_index(drop=True)
        y_left = y[left_mask]
        y_right = y[~left_mask]

        node.left = self._build(X_left, y_left, depth + 1)
        node.right = self._build(X_right, y_right, depth + 1)
        return node

    # ---------- Public API ----------
    def fit(self, X, y):
        X = X.reset_index(drop=True)
        y = np.asarray(y)
        if self.task == "classification":
            self._classes = np.unique(y)
        else:
            y = y.astype(float)
        self.root = self._build(X, y, depth=0)
        return self

    def _predict_row(self, row, node):
        # Leaves are the only nodes with no children.
        if node.left is None:
            return node.prediction
        if node.is_numeric:
            go_left = row[node.feature] <= node.threshold
        else:
            go_left = row[node.feature] in node.left_categories
        return self._predict_row(row, node.left if go_left else node.right)

    def predict(self, X):
        return np.array([self._predict_row(row, self.root)
                         for _, row in X.iterrows()])


# Backwards-compatible alias so the original COMPAS cell still works.
def CARTClassifier(**kw):
    return CART(task="classification", **kw)

In [5]:
# ============================================================
# Demo 1 — Binary classification on COMPAS (is_recid)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

tree = CART(
    task="classification",
    max_depth=6,
    min_samples_split=40,
    min_samples_leaf=20,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Custom CART test accuracy: {acc:.4f}")

# Quick baseline reference: predict the majority class.
majority = pd.Series(y_train).mode()[0]
baseline = accuracy_score(y_test, np.full_like(y_test, majority))
print(f"Majority-class baseline:   {baseline:.4f}")

Train: (13958, 8)  Test: (3490, 8)


Custom CART test accuracy: 0.6713
Majority-class baseline:   0.5201


In [6]:
# ============================================================
# Demo 2 — Multiclass classification on the Iris dataset
# ============================================================
# Uses sklearn ONLY to load the dataset and split — the model itself is
# still our from-scratch CART.
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X_iris = iris.data                  # 4 numeric features
y_iris = iris.target.values         # 3 classes: 0, 1, 2

iris_numeric = list(X_iris.columns)
iris_categorical = []               # iris has no categorical features

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.25, stratify=y_iris, random_state=0
)

iris_tree = CART(
    task="classification",
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    numeric_features=iris_numeric,
    categorical_features=iris_categorical,
)
iris_tree.fit(X_tr, y_tr)

iris_pred = iris_tree.predict(X_te)
iris_acc = accuracy_score(y_te, iris_pred)
print(f"Iris (3-class) test accuracy: {iris_acc:.4f}")
print("Predicted class distribution:", dict(zip(*np.unique(iris_pred, return_counts=True))))

Iris (3-class) test accuracy: 0.9474
Predicted class distribution: {np.int64(0): np.int64(13), np.int64(1): np.int64(15), np.int64(2): np.int64(10)}


In [7]:
# ============================================================
# Demo 3 — Regression on the Diabetes dataset
# ============================================================
# Predict a continuous "disease progression" score from 10 numeric features.
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, r2_score

diab = load_diabetes(as_frame=True)
X_d = diab.data                # 10 numeric features
y_d = diab.target.values       # continuous target

diab_numeric = list(X_d.columns)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_d, y_d, test_size=0.25, random_state=0
)

reg_tree = CART(
    task="regression",
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    numeric_features=diab_numeric,
    categorical_features=[],
)
reg_tree.fit(X_tr, y_tr)

y_hat = reg_tree.predict(X_te)
rmse = mean_squared_error(y_te, y_hat) ** 0.5
r2 = r2_score(y_te, y_hat)
mean_baseline_rmse = mean_squared_error(
    y_te, np.full_like(y_te, y_tr.mean(), dtype=float)
) ** 0.5
print(f"Diabetes regression — RMSE: {rmse:.2f}  R^2: {r2:.4f}")
print(f"Mean-prediction baseline RMSE: {mean_baseline_rmse:.2f}")

Diabetes regression — RMSE: 67.72  R^2: 0.0763
Mean-prediction baseline RMSE: 70.46
